In [1]:
import kagglehub
import torch
import os
from torch import nn
import torch.optim as optim
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, DataLoader
from torchvision import transforms, datasets

path = kagglehub.dataset_download("muratkokludataset/rice-image-dataset")

100%|██████████| 219M/219M [00:15<00:00, 14.7MB/s]

Extracting files...


In [2]:
data = os.path.join(path,'Rice_Image_Dataset')

In [3]:
transform = transforms.Compose([transforms.Resize((128, 128)),
                               transforms.ToTensor()]) #kinda of a normalization method
dataset = ImageFolder(root=data, transform=transform)

In [4]:
train_size = int(0.9*len(dataset))
test_size = len(dataset) - train_size

In [5]:
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [6]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [7]:
from torch.nn.modules.activation import Softmax
class RiceClassification(nn.Module):
  def __init__(self):
    super(RiceClassification, self).__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(3*128*128, 264),
        nn.ReLU(),
        nn.Linear(264, 5),
        nn.LogSoftmax(dim=1),
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits



In [8]:
def get_accuracy(pred, labels):
  _,predictions = torch.max(pred, 1)
  correct = (predictions == labels).float().sum()
  accuracy = correct/labels.shape[0]
  return accuracy

In [12]:
model = RiceClassification()

loss_fun = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

def train(dataloader, model, loss_fun, optimizer):
  size = len(dataloader.dataset)
  model.train()
  train_loss, accuracy_loss = 0, 0
  for batch, (X,y) in enumerate(dataloader):
    optimizer.zero_grad()
    pred = model(X)
    loss = loss_fun(pred, y)

    loss.backward()
    optimizer.step()

    accuracy = get_accuracy(pred, y)
    train_loss += loss.item()
    accuracy_loss += accuracy.item()

    if batch % 100 == 0:
      current = batch*len(X)
      avg_loss = train_loss / (batch+1)
      avg_accuracy = accuracy_loss / (batch+1)

      print(f'Batch {batch}, Loss {avg_loss:>7f}, '
            f'Accuracy {avg_accuracy:>0.2f}, '
            f'[{current:>5d}/{size:>5d}]')

epochs = 2
for t in range(epochs):
  print(f'Epochs {t+1}\n................................')
  train(train_loader, model ,loss_fun, optimizer)
print('Done!')



Epochs 1
................................
Batch 0, Loss 1.617444, Accuracy 0.23, [    0/67500]
Batch 100, Loss 0.427898, Accuracy 0.88, [ 6400/67500]
Batch 200, Loss 0.295518, Accuracy 0.91, [12800/67500]
Batch 300, Loss 0.245949, Accuracy 0.92, [19200/67500]
Batch 400, Loss 0.212819, Accuracy 0.93, [25600/67500]
Batch 500, Loss 0.191685, Accuracy 0.94, [32000/67500]
Batch 600, Loss 0.179020, Accuracy 0.94, [38400/67500]
Batch 700, Loss 0.168528, Accuracy 0.95, [44800/67500]
Batch 800, Loss 0.159622, Accuracy 0.95, [51200/67500]
Batch 900, Loss 0.152258, Accuracy 0.95, [57600/67500]
Batch 1000, Loss 0.147058, Accuracy 0.95, [64000/67500]
Epochs 2
................................
Batch 0, Loss 0.071677, Accuracy 0.95, [    0/67500]
Batch 100, Loss 0.095843, Accuracy 0.96, [ 6400/67500]
Batch 200, Loss 0.090485, Accuracy 0.97, [12800/67500]
Batch 300, Loss 0.087059, Accuracy 0.97, [19200/67500]
Batch 400, Loss 0.087377, Accuracy 0.97, [25600/67500]
Batch 500, Loss 0.084665, Accuracy 0.97

In [13]:
def test(dataloader, model):
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  model.eval()
  test_loss, correct = 0, 0
  with torch.no_grad():
    for X, y in dataloader:
      pred = model(X)
      test_loss += loss_fun(pred, y).item()
      correct += (pred.argmax(1)==
                  y).type(torch.float).sum().item()

  test_loss /= num_batches
  correct /= size

  print(f'Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n')


In [14]:
test(test_loader, model)

Test Error: 
 Accuracy: 97.8%, Avg loss: 0.060408 



In [21]:
import matplotlib.pyplot as plt

def predict_single_image(image, label, model):
  model.eval()
  image = image.unsqueeze(0)

  with torch.no_grad():
    prediction = model(image)
    print(prediction)
    predicted_label = prediction.argmax(1).item()

  plt.imshow(image.squeeze().permute(1, 2, 0))
  plt.title(f'Predicted: {predicted_label}, Actual: {label}')
  plt.show()

  return predicted_label